In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
import re
from unidecode import unidecode

# ---------- Config ----------
final_folder = Path("Final")

# ---------- Normalizador ----------
def normalize_text(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    s = unidecode(s)                      # quita acentos
    s = re.sub(r"[^\w\s]", "_", s)        # puntos, guiones, etc. -> "_"
    s = re.sub(r"\s+", "_", s)            # espacios -> "_"
    s = re.sub(r"_+", "_", s)             # colapsa múltiples "_"
    return s.strip("_")                   # quita "_" al inicio/fin

def normalize_columns(df):
    df = df.copy()
    # renombra columnas
    df.columns = [normalize_text(c) for c in df.columns]
    # normaliza valores string/objeto
    for c in df.select_dtypes(include="object").columns:
        df[c] = df[c].map(normalize_text)
    return df


In [ ]:
raw_folder = Path("Raw")

# Eco Bici


In [8]:
CSV_ESTACIONES  = pd.read_csv(raw_folder / "ecobici_estaciones.csv")


In [9]:
CSV_ESTACIONES 

,station_id,name,lat,lon,capacity
0,1,CE-710 Molino del Rey - Glorieta de la Lealtad,19.416795,-99.192508,39
1,10,CE-438 Adolfo Prieto-José María Olloqui,19.364730,-99.174447,23
2,100,CE-105 San Jerónimo-5 de Febrero,19.426755,-99.135014,23
3,101,CE-102 Echeveste-Bolivar,19.428214,-99.139494,15
4,102,CE-097 República de Salvador-Pino Suárez,19.429189,-99.132759,27
...,...,...,...,...,...
672,95,CE-060 Mazatlán - Fernando Montes de Oca,19.414099,-99.177158,39
673,96,CE-302 Luz Saviñón-Magdalena,19.393914,-99.171675,39
674,97,CE-070 Parque México - Michoacán,19.411280,-99.169825,45
675,98,CE-125 Morelia-Circular de Morelia,19.424197,-99.156127,35


In [10]:
gdf_estaciones = gpd.GeoDataFrame(
    CSV_ESTACIONES,
    geometry=gpd.points_from_xy(CSV_ESTACIONES['lon'], CSV_ESTACIONES['lat']),
    crs="EPSG:4326"
)
gdf_estaciones_6372 = gdf_estaciones.to_crs(epsg=6372)

In [13]:
gdf_estaciones_6372 = gdf_estaciones_6372.rename(columns={'station_id': 'id', 'capacity': 'var', 'geometry': 'geom'})

In [18]:
eco = gdf_estaciones_6372[['id', 'var', 'geom']]

In [ ]:
eco

,id,var,geom
0,1,39,POINT (2793988.533 827190.535)
1,10,23,POINT (2795992.773 821481.897)
2,100,23,POINT (2799986.306 828408.715)
3,101,15,POINT (2799514.066 828560.356)
4,102,27,POINT (2800217.02 828682.031)
...,...,...,...
672,95,39,POINT (2795601.518 826924.603)
673,96,39,POINT (2796219.627 824708.367)
674,97,45,POINT (2796375.504 826628.612)
675,98,35,POINT (2797781.5 828082.432)


In [39]:
OUT_ECO   =  Path(final_folder / "ecobici.parquet")
eco = eco.set_geometry("geom")

eco.to_parquet(OUT_ECO, engine="pyarrow")

# Ciclovia

In [26]:
PARQUET_INFRA  = gpd.read_parquet(raw_folder / "infraestructura_ciclista.parquet")

In [28]:
PARQUET_INFRA = PARQUET_INFRA[['id_tramo','tipo_ic', 'geom']]

In [29]:
PARQUET_INFRA.head()

,id_tramo,tipo_ic,geom
0,111,carril_de_prioridad_ciclista,"LINESTRING (-99.15975 19.42639, -99.16195 19.4..."
1,116,carril_de_prioridad_ciclista,"LINESTRING (-99.16092 19.42098, -99.16077 19.4..."
2,160,carril_de_prioridad_ciclista,"LINESTRING (-99.17124 19.37789, -99.17108 19.3..."
3,178,carril_de_prioridad_ciclista,"LINESTRING (-99.15428 19.43551, -99.15352 19.4..."
4,179,carril_de_prioridad_ciclista,"LINESTRING (-99.15399 19.4367, -99.154 19.4367..."


In [30]:
PARQUET_INFRA["tipo_ic"].value_counts()

tipo_ic
ciclovia                              462
ciclocarril                           200
carril_bus_bici                       196
sendero_compartido                    170
ciclovia_bidireccional                122
carril_de_prioridad_ciclista           84
infraestructura_ciclista_emergente     68
Name: count, dtype: int64

In [32]:
PARQUET_INFRA_6372 = PARQUET_INFRA.to_crs(epsg=6372)

In [33]:
PARQUET_INFRA_6372['var'] = PARQUET_INFRA_6372['tipo_ic'].map(
    lambda x: 'prot' if x in [
        'ciclovia', 
        'ciclovia_bidireccional', 
        'carril_bus_bici', 
        'sendero_compartido'
    ] else (
        'comp' if x in [
            'ciclocarril', 
            'carril_de_prioridad_ciclista', 
            'infraestructura_ciclista_emergente'
        ] else None
    )
)

In [35]:
via = PARQUET_INFRA_6372[['id_tramo', 'var', 'geom']].rename(columns={'id_tramo': 'id'})

In [36]:
via

,id,var,geom
0,111,comp,"LINESTRING (2797397.682 828317.102, 2797169.11..."
1,116,comp,"LINESTRING (2797287.215 827718.029, 2797303.09..."
2,160,comp,"LINESTRING (2796300.021 822940.607, 2796316.70..."
3,178,comp,"LINESTRING (2797949.813 829334.756, 2798029.66..."
4,179,comp,"LINESTRING (2797977.838 829466.629, 2797977.13..."
...,...,...,...
1297,616,prot,"LINESTRING (2793881.098 827308.411, 2793882.17..."
1298,617,prot,"LINESTRING (2793864.643 827910.037, 2793863.44..."
1299,618,prot,"LINESTRING (2793864.643 827910.037, 2793864.06..."
1300,627,prot,"LINESTRING (2804018.211 816730.365, 2804020.74..."


In [38]:
via = via.set_geometry("geom")
via.to_parquet(final_folder / "via.parquet", engine="pyarrow")
